In [4]:
import pandas as pd
import numpy as nmp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier,  RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression,  Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import mean_absolute_error, explained_variance_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [5]:
encounters_df = pd.read_csv('nationwide-encounters-fy21-fy24-aor.csv')

In [6]:
encounters_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58744 entries, 0 to 58743
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Fiscal Year             58744 non-null  int64 
 1   Month Grouping          58744 non-null  object
 2   Month (abbv)            58744 non-null  object
 3   Component               58744 non-null  object
 4   Land Border Region      58744 non-null  object
 5   Area of Responsibility  58744 non-null  object
 6   AOR (Abbv)              58744 non-null  object
 7   Demographic             58744 non-null  object
 8   Citizenship             58744 non-null  object
 9   Title of Authority      58744 non-null  object
 10  Encounter Type          58744 non-null  object
 11  Encounter Count         58744 non-null  int64 
dtypes: int64(2), object(10)
memory usage: 5.4+ MB


In [7]:
encounters_df.describe()

,Fiscal Year,Encounter Count
count,58744.000000,58744.000000
mean,2022.578731,184.280727
std,1.084640,777.223187
min,2021.000000,1.000000
25%,2022.000000,3.000000
50%,2023.000000,10.000000
75%,2024.000000,67.000000
max,2024.000000,25457.000000


In [8]:
encounters_df.head()

,Fiscal Year,Month Grouping,Month (abbv),Component,Land Border Region,Area of Responsibility,AOR (Abbv),Demographic,Citizenship,Title of Authority,Encounter Type,Encounter Count
0,2021,FYTD,APR,Office of Field Operations,Northern Land Border,Boston Field Office,Boston,Accompanied Minors,CANADA,Title 8,Inadmissibles,3
1,2021,FYTD,APR,Office of Field Operations,Northern Land Border,Boston Field Office,Boston,Accompanied Minors,MEXICO,Title 42,Expulsions,2
2,2021,FYTD,APR,Office of Field Operations,Northern Land Border,Boston Field Office,Boston,Accompanied Minors,OTHER,Title 42,Expulsions,6
3,2021,FYTD,APR,Office of Field Operations,Northern Land Border,Boston Field Office,Boston,Single Adults,CANADA,Title 42,Expulsions,11
4,2021,FYTD,APR,Office of Field Operations,Northern Land Border,Boston Field Office,Boston,Single Adults,CANADA,Title 8,Inadmissibles,40


In [11]:
X = ['Citizenship', 'Demographic', 'Encounter Type', 'Area of Responsibility', 'Fiscal Year']
y = 'Encounter Count'
X_train, X_test, y_train, y_test = train_test_split(encounters_df[X], encounters_df[y], test_size=0.2, random_state=42)

In [14]:
# Update categorical features
categorical_features = ['Citizenship', 'Demographic', 'Encounter Type', 'Area of Responsibility']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

In [19]:
from itertools import product

# Filter the dataframe for Fiscal Year 2025
future_df = encounters_df[encounters_df['Fiscal Year'] == 2025]

# If there is no data for 2025, create all combinations of categorical features for 2025
if future_df.empty:

    unique_values = {col: encounters_df[col].unique() for col in categorical_features}
    combinations = list(product(*[unique_values[col] for col in categorical_features]))
    future_df = pd.DataFrame(combinations, columns=categorical_features)
    future_df['Fiscal Year'] = 2025

# Predict Encounter Count for 2025 by categorical feature combination
future_X = future_df[X]
future_df['Predicted Encounter Count'] = pipeline.predict(future_X)

# Show predictions grouped by categorical features
grouped = future_df.groupby(categorical_features)['Predicted Encounter Count'].sum().reset_index()
print(grouped)

      Citizenship         Demographic Encounter Type  Area of Responsibility  \
0          BRAZIL  Accompanied Minors  Apprehensions    Atlanta Field Office   
1          BRAZIL  Accompanied Minors  Apprehensions  Baltimore Field Office   
2          BRAZIL  Accompanied Minors  Apprehensions         Big Bend Sector   
3          BRAZIL  Accompanied Minors  Apprehensions           Blaine Sector   
4          BRAZIL  Accompanied Minors  Apprehensions     Boston Field Office   
...           ...                 ...            ...                     ...   
10819   VENEZUELA  UC / Single Minors  Inadmissibles          Swanton Sector   
10820   VENEZUELA  UC / Single Minors  Inadmissibles      Tampa Field Office   
10821   VENEZUELA  UC / Single Minors  Inadmissibles     Tucson Field Office   
10822   VENEZUELA  UC / Single Minors  Inadmissibles           Tucson Sector   
10823   VENEZUELA  UC / Single Minors  Inadmissibles             Yuma Sector   

       Predicted Encounter Count  
0   

In [20]:
# Build and fit the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Citizenship', 'Demographic',
                                                   'Encounter Type',
                                                   'Area of '
                                                   'Responsibility'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [21]:
# Evaluate the model
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
r2 = r2_score(y_test, y_pred)
print(f"R^2 Score: {r2}")

Mean Squared Error: 160980.62770772044
R^2 Score: 0.6969168918168149


In [22]:
# Assess model evaluation quality
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

if r2 > 0.7:
    print("The model explains a large portion of the variance in the data (good fit).")
elif r2 > 0.5:
    print("The model explains a moderate portion of the variance (reasonable fit).")
else:
    print("The model explains little variance (poor fit).")

print("Lower MSE values indicate better predictive accuracy. Compare this to the scale of your target variable for context.")

Mean Squared Error (MSE): 160980.63
R^2 Score: 0.70
The model explains a moderate portion of the variance (reasonable fit).
Lower MSE values indicate better predictive accuracy. Compare this to the scale of your target variable for context.


In [23]:
# Regression metrics report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

Mean Absolute Error: 98.48
Mean Squared Error: 160980.63
R^2 Score: 0.70


In [26]:
import joblib

# Provide metadata about the model
model_metadata = {
    'model_type': 'Random Forest Regressor',
    'features': X,
    'target': y,
    'training_data_size': len(X_train),
    'test_data_size': len(X_test),
    'mean_squared_error': mse,
    'r2_score': r2,
    'mean_absolute_error': mae,
    'model_parameters': pipeline.get_params(),
    'training_date': pd.Timestamp.now().isoformat(),
    'version': '1.0',
    'author': 'Exeario Boscan',
    'description': 'Random Forest Regressor model for predicting encounter counts based on categorical features.'
}
# Save the model and metadata
joblib.dump(pipeline, 'random_forest_regressor_model_V1.pkl')

['random_forest_regressor_model_V1.pkl']